In [ ]:
import requests
import pandas as pd
import time

BASE_URL = "https://mobidb.org/api/download_page"

COMMON_PARAMS = {
    "limit": 100,
    "projection": "level,acc,name,organism,length,"
                  "prediction-disorder-mobidb_lite.content_fraction,"
                  "prediction-disorder-alphafold.content_fraction",
    "prediction-disorder-mobidb_lite.content_fraction_min": 0.5,
    "prediction-disorder-mobidb_lite.content_fraction_max": 0.7,
    "sort_asc": "_id",
}

MAX_RECORDS = 3_100_000
TIMEOUT     = 60
HEADERS     = {
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://mobidb.org/browse",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest",
}

all_results = []
id_gt       = None
page        = 0

while len(all_results) < MAX_RECORDS:
    params = {**COMMON_PARAMS}
    if id_gt:
        params["_id_gt"] = id_gt

    for attempt in range(3):
        try:
            r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
            break
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** attempt)
    else:
        print("3 failed attempts, stopping.")
        break

    records = body.get("data", [])
    if not records:
        print("No more records.")
        break

    all_results.extend(records)
    page += 1
    print(f"\rPage {page} | {len(all_results):,} / {MAX_RECORDS:,}", end="", flush=True)

    # cursor for next page comes from metadata
    id_gt = body.get("metadata", {}).get("last_id")
    if not id_gt:
        print("\nNo next cursor, reached last page.")
        break

    if len(records) < COMMON_PARAMS["limit"]:
        print("\nPartial page, reached last page.")
        break

print()
df = pd.DataFrame(all_results[:MAX_RECORDS])
print(df.head())
df.to_csv("mobidb_full_results.csv", index=False)


Page 31000 | 3,100,000 / 3,100,000
      acc                          name  length  \
0  Q197C8  Uncharacterized protein 032R     279   
1  Q6GZU4  Uncharacterized protein 032R     629   
2  Q196X1  Uncharacterized protein 089L      98   
3  Q6GZN2  Uncharacterized protein 093L      55   
4  Q6GZN6   Uncharacterized protein 89R     388   

                                            organism  \
0  Invertebrate iridescent virus 3 (IIV-3) (Mosqu...   
1               Frog virus 3 (isolate Goorha) (FV-3)   
2  Invertebrate iridescent virus 3 (IIV-3) (Mosqu...   
3               Frog virus 3 (isolate Goorha) (FV-3)   
4               Frog virus 3 (isolate Goorha) (FV-3)   

  prediction-disorder-mobidb_lite  level prediction-disorder-alphafold  
0     {'content_fraction': 0.516}    NaN                           NaN  
1      {'content_fraction': 0.61}    NaN                           NaN  
2     {'content_fraction': 0.643}    NaN                           NaN  
3     {'content_fraction': 0.

In [40]:
import validators


levels_good = []
seq_list = []
for protein in df.itertuples():
    if protein.level >= 2.0:
        levels_good.append(protein.acc)
        seq_list.append(protein)

valid_urls = []

for protein in levels_good:
    url = f"https://alphafold.ebi.ac.uk/files/AF-{protein}-F1-model_v6.pdb"
    if validators.url(url):
        valid_urls.append(url)
    else:
        print(f"Invalid URL: {url}")

valid_urls = pd.DataFrame(valid_urls, columns=["Valid_API_URL"])
with open("urls.txt", "w") as f:
    for url in valid_urls["Valid_API_URL"]:
        f.write(url + "\n")


In [38]:
seq_list = pd.DataFrame(seq_list)
seq_list.to_csv("mobidb_filtered_data.csv", index=False)

In [39]:
seq_list.head()

,Index,acc,name,length,organism,_5,level,_7
0,9,Q06649,SH3 domain-binding protein 2,559,Mus musculus (Mouse),0.512,2.0,0.555
1,10,Q0P5A7,Eukaryotic translation initiation factor 4E-bi...,118,Bos taurus (Bovine),0.653,2.0,0.720
2,18,Q9NYB9,Abl interactor 2,513,Homo sapiens (Human),0.517,2.0,0.665
3,26,P33379,Actin assembly-inducing protein,639,Listeria monocytogenes serovar 1/2a (strain AT...,0.607,2.0,0.915
4,33,A2AM29,Protein AF-9,569,Mus musculus (Mouse),0.596,2.0,0.634
